# EfficientNet-B0 on PlantVillage (adapted from Lab 2)
Run on Colab GPU (Runtime > Change runtime type > T4). Owner: Group 1 (Omkar).
Dataset folder = one sub-folder per class (full PlantVillage, e.g. `Tomato___Late_blight`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/plantvillage'          # <- change to your dataset folder
OUT  = '/content/drive/MyDrive/agri_ai_models'        # checkpoints survive disconnects
import os; os.makedirs(OUT, exist_ok=True)

In [ ]:
import json, random, time
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
def seed_everything(seed=SEED):                      # from Lab 2
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
tf_train = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)), transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20), transforms.ColorJitter(0.3, 0.3, 0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
tf_eval = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

base_train = datasets.ImageFolder(ROOT, transform=tf_train)
base_eval  = datasets.ImageFolder(ROOT, transform=tf_eval)
CLASSES = base_train.classes
targets = np.array(base_train.targets)
idx = np.arange(len(targets))
# 70 / 15 / 15 stratified split. NOTE: PlantVillage has several photos per leaf, so a random
# split can leak and inflate accuracy -> report PlantDoc results as the honest check.
tr, tmp = train_test_split(idx, test_size=0.30, stratify=targets, random_state=SEED)
va, te  = train_test_split(tmp, test_size=0.50, stratify=targets[tmp], random_state=SEED)
mk = lambda ds, ix, sh: DataLoader(Subset(ds, ix), batch_size=32, shuffle=sh, num_workers=2, pin_memory=True)
train_loader, val_loader, test_loader = mk(base_train, tr, True), mk(base_eval, va, False), mk(base_eval, te, False)
json.dump(CLASSES, open(f'{OUT}/classes.json', 'w'))
print(len(CLASSES), 'classes | train', len(tr), 'val', len(va), 'test', len(te))

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device=DEVICE):      # from Lab 2
    model.train(); tl, c, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad(); logits = model(X); loss = criterion(logits, y); loss.backward(); optimizer.step()
        tl += loss.item() * len(y); c += (logits.argmax(1) == y).sum().item(); n += len(y)
    return tl / n, c / n

@torch.no_grad()
def eval_epoch(model, loader, criterion, device=DEVICE):                  # from Lab 2
    model.eval(); tl, c, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X); loss = criterion(logits, y)
        tl += loss.item() * len(y); c += (logits.argmax(1) == y).sum().item(); n += len(y)
    return tl / n, c / n

In [ ]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CLASSES))
model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss()
best, hist = 0.0, {'train': [], 'val': []}

def run(epochs, lr, tag, cosine=False):
    global best
    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs) if cosine else None
    for e in range(epochs):
        t0 = time.time()
        _, tra = train_epoch(model, train_loader, opt, criterion)
        _, vaa = eval_epoch(model, val_loader, criterion)
        if sch: sch.step()
        hist['train'].append(tra); hist['val'].append(vaa)
        if vaa > best:
            best = vaa; torch.save(model.state_dict(), f'{OUT}/efficientnet_b0.pt')
        print(f'{tag} {e+1}/{epochs} | train {tra:.3f} | val {vaa:.3f} | best {best:.3f} | {time.time()-t0:.0f}s')

# Stage 1: head only
for p in model.features.parameters(): p.requires_grad = False
run(3, 1e-3, 'head')
# Stage 2: fine-tune everything
for p in model.parameters(): p.requires_grad = True
run(6, 1e-4, 'finetune', cosine=True)

In [ ]:
# Final TEST metrics (touch the test set only once, here)
model.load_state_dict(torch.load(f'{OUT}/efficientnet_b0.pt')); model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for X, y in test_loader:
        y_pred += model(X.to(DEVICE)).argmax(1).cpu().tolist(); y_true += y.tolist()
rep = classification_report(y_true, y_pred, target_names=CLASSES, digits=3, output_dict=True)
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))
json.dump(rep, open(f'{OUT}/test_metrics.json', 'w'), indent=1)
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12, 10)); plt.imshow(cm, cmap='Blues'); plt.colorbar()
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion matrix (test)')
plt.savefig(f'{OUT}/confusion_matrix.png', dpi=150, bbox_inches='tight'); plt.show()
print('Download efficientnet_b0.pt and classes.json from', OUT, 'into repo /models')

## Optional: PlantDoc (real-world) check
Evaluate the same model on PlantDoc images of classes that exist in both datasets
(map folder names to PlantVillage class names) and report the accuracy drop honestly.